# Second learned-identity experiment

Select a GPU runtime first: **Runtime > Change runtime type**, pick the best GPU offered (A100 if available).
Then run the first code cell once. It starts everything in the background. Run the second cell to watch progress.

What it does: freezes the pinned CS-NET v4 base, gives an identity to every player seen in 2 or more
non-held-back matches, trains only the identity adapter, and runs the likelihood "who is it?" test once
on the held-back matches. The pass rule is fixed in `clone-model/v2.py`. See `SPEC.md` section 8.9.

It does not install or control CS2, and it does not generate play. Data: Julien Blanchon / OpenCS2 (CC BY 4.0).

In [ ]:
# Second learned-identity experiment (clone-model/v2.py). Run this ONCE on a GPU runtime.
# It starts setup, data preparation, training and the held-out test in the background and returns at once.
# Watch progress with the next cell. Everything is logged to /content/run2/log.txt.
import os, subprocess, pathlib, torch
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type, pick the best GPU (A100 if offered), then run again."
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, round(gpu.total_memory / 1024**3), "GB")
BATCH = 6 if gpu.total_memory >= 32 * 1024**3 else 3      # windows per step; each window has up to 10 targets
ROOT = pathlib.Path("/content/run2"); ROOT.mkdir(exist_ok=True)
script = f"""
set -euo pipefail
cd {ROOT}
PROJECT_REV=94a6fdb
CSNET_REV=268dd2193c8a3d6e330c07dde5a851196c94eeaa
echo "STAGE setup $(date -u +%H:%M:%S)"
pip install -q webdataset zstandard open3d demoparser2 pandas pyyaml huggingface_hub pytest
[ -d csnet ] || git clone -q https://github.com/Gary2005/cs-net.git csnet
git -C csnet checkout -q $CSNET_REV
[ -d project ] || git clone -q https://github.com/OhamjDung/HackUTA-CS2.git project
git -C project fetch -q origin && git -C project checkout -q $PROJECT_REV
(cd csnet && python scripts/download_checkpoints.py --pretrain-only --out-dir {ROOT}/checkpoints)
[ -f data.zip ] || curl -sSL -o data.zip https://github.com/OhamjDung/HackUTA-CS2/releases/download/clone-training-data-v1/cs2-rounds-transfer.zip
echo "0e46914cbca514efbb5727b64f9e753116da4afaa98f764be66d15d97cb8ad25  data.zip" | sha256sum -c -
unzip -q -o data.zip
echo "STAGE tests $(date -u +%H:%M:%S)"
(cd project/clone-model && CSNET_ROOT={ROOT}/csnet python -m pytest tests -q -x 2>&1 | tail -3)
echo "STAGE prepare $(date -u +%H:%M:%S)"
python project/clone-model/prepare.py --csnet csnet --json-dir round_json --games project/clone-model/games.json --out cache | tail -3
COMMON="--csnet csnet --cache cache --checkpoint checkpoints/cs-net-v4-pro.pt --out out --device cuda"
echo "STAGE train $(date -u +%H:%M:%S)"
python project/clone-model/v2.py train $COMMON --steps 2000 --batch-size {BATCH}
echo "STAGE test $(date -u +%H:%M:%S)"
python project/clone-model/v2.py test $COMMON
echo "STAGE done $(date -u +%H:%M:%S)"
"""
(ROOT / "run.sh").write_text(script)
if (ROOT / "pid").exists() and os.path.exists(f"/proc/{(ROOT / 'pid').read_text().strip()}"):
    print("Already running. Use the next cell to watch it.")
else:
    log = open(ROOT / "log.txt", "a")
    process = subprocess.Popen(["bash", str(ROOT / "run.sh")], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    (ROOT / "pid").write_text(str(process.pid))
    print("Started in the background, process", process.pid, "| batch", BATCH)

In [ ]:
# Progress of the background run. Safe to run as often as you like.
import os, pathlib, json
ROOT = pathlib.Path("/content/run2")
pid = (ROOT / "pid").read_text().strip() if (ROOT / "pid").exists() else None
print("running:", bool(pid) and os.path.exists(f"/proc/{pid}"))
lines = (ROOT / "log.txt").read_text(errors="replace").splitlines() if (ROOT / "log.txt").exists() else []
for line in [l for l in lines if l.startswith(("STAGE", "TRAINING_", "DEV", "EARLY_STOP", "TEST_COMPLETE"))][-14:]:
    print(line[:400])
print("--- last lines")
for line in lines[-6:]:
    print(line[:300])
for name in ("progress.json", "test.json"):
    path = ROOT / "out" / name
    if path.exists():
        data = json.loads(path.read_text())
        print(name, {k: v for k, v in data.items() if k not in ("detail", "ranks", "candidates")})